# 01 — Data Generation

**NovaStream Customer Retention Intelligence Platform**

This notebook documents how the synthetic NovaStream dataset was generated.

The full generation logic lives in `src/generate_data.py`.

Design principles used in generation:

- Churn is driven by a latent risk score built from acquisition channel,
  plan, billing cycle, age, discount, and customer-level random variation.
- Engagement, satisfaction, and support-ticket behavior are correlated with
  the same underlying risk plus independent noise.
- Realistic data-quality problems such as missing values, duplicates,
  inconsistent labels, and orphan foreign keys are injected into the raw data.


In [1]:

result = run_script("generate_data.py")

if result.returncode != 0:
    print("Data generation failed.")


Step 1/8: Generating customers...
Step 2/8: Assigning plans & simulating churn (latent-risk survival model)...
Step 3/8: Simulating reactivations (repeat subscriptions)...
  customers: 52,000 | subscriptions: 52,563
Step 4/8: Generating transactions...
  transactions: 394,646
Step 5/8: Generating customer activity records...
  customer_activity: 976,631
Step 6/8: Generating support tickets...
  support_tickets: 78,233
Step 7/8: Generating marketing campaigns & interactions...
  marketing_campaigns: 180 | interactions: 103,769
Step 8/8: Injecting realistic data quality issues...
Writing raw CSVs to data/raw/ ...

=== RAW DATA GENERATION COMPLETE ===
  customers                               52,520 rows
  plans                                        4 rows
  subscriptions                           52,563 rows
  transactions                           397,803 rows
  customer_activity                      976,631 rows
  support_tickets                         78,233 rows
  marketing_campaig

## Inspect the generated raw tables

In [2]:

import pandas as pd

RAW = ROOT / "data" / "raw"

tables = {}

for name in [
    "customers",
    "plans",
    "subscriptions",
    "transactions",
    "customer_activity",
    "support_tickets",
    "marketing_campaigns",
    "customer_campaign_interactions"
]:
    tables[name] = pd.read_csv(RAW / f"{name}.csv")
    print(f"{name:35s} shape={tables[name].shape}")


customers                           shape=(52520, 8)
plans                               shape=(4, 6)
subscriptions                       shape=(52563, 11)
transactions                        shape=(397803, 8)
customer_activity                   shape=(976631, 7)
support_tickets                     shape=(78233, 7)
marketing_campaigns                 shape=(180, 5)
customer_campaign_interactions      shape=(103769, 5)


In [3]:

tables["customers"].head()


In [4]:

print("customers.csv head:")
print(tables["customers"].head().to_string())


customers.csv head:
   customer_id signup_date  gender   age        country         region acquisition_channel   referral_source
0            1  2025-04-21    Male  38.0  UNITED STATES  North America        Social Media               NaN
1            2  2024-09-10  Female  48.0         FRANCE         Europe      Organic Search  Blog/Review Site
2            3  2025-06-11    Male  50.0         Brazil  Latin America         Paid Search        Influencer
3            4  2025-03-04    Male  38.0        Germany         Europe           Affiliate     Search Engine
4            5  2023-12-03  Female  33.0         Canada  North America     Email Marketing         App Store


In [5]:

print("Sample of injected data-quality problems:")

print(
    "- Duplicate customer_id count:",
    tables["customers"]["customer_id"].duplicated().sum()
)

print(
    "- Missing ages:",
    tables["customers"]["age"].isna().sum()
)

print(
    "- Ages out of plausible range:",
    (
        (tables["customers"]["age"] < 0)
        |
        (tables["customers"]["age"] > 100)
    ).sum()
)

print(
    "- Upper-case country values:",
    tables["customers"]["country"]
    .dropna()
    .apply(lambda x: isinstance(x, str) and x.isupper())
    .sum()
)


Sample of injected data-quality problems:
- Duplicate customer_id count: 520
- Missing ages: 1085
- Ages out of plausible range: 196
- Upper-case country values: 2584


In [6]:

subs = tables["subscriptions"]

churn_rate = (
    subs["subscription_status"]
    .str.lower()
    .eq("cancelled")
    .mean()
)

print(f"Raw subscription-level churn rate: {churn_rate:.2%}")

print(
    subs["subscription_status"].value_counts()
)


Raw subscription-level churn rate: 21.49%
subscription_status
Active       40863
Cancelled    11175
active         403
cancelled      122
Name: count, dtype: int64
